# Part 20 — Conversational RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-20-conversational-rag/conversational_rag.ipynb)

*Part 19's agent answered one question and stopped — it had no memory of anything said before. A real chat app gets **follow-ups**: "What about damaged items?", "And how long does that take?" — fragments that mean nothing on their own. This part gives the pipeline a memory and runs it, **by hand**.*

📖 Read the essay: https://www.mefby.com/essays/conversational-rag

🐍 Companion script: `conversational_rag.py`

## What you'll build

Sent to a retriever as-is, *"What about damaged items?"* matches no useful chunk: it never mentions refunds, so it can't find the refund clause. The fix is not a bigger index — it's **reading the conversation before you retrieve**. That move is **query condensation** (history-aware query rewriting): rewrite a context-dependent follow-up into a **self-contained, standalone query** using the conversation so far. The standalone query is what hits the index; the raw fragment never does.

Concretely you'll build, cell by cell:

- the support knowledge base carried over from Parts 6–12 (refunds, the **E-4042** error, shipping, warranty), with two **new** clauses so turn 2 has a real target chunk **and** a deliberate distractor;
- a transparent offline retriever (the series' lexical stand-in, the reproducible default);
- **conversation memory** — a rolling transcript of `(user, assistant)` turns the condenser reads;
- the `generate()` provider shape for the **real** LLM-driven condenser, plus a deterministic rule-based condenser that is the offline source of truth;
- the **condense → retrieve → answer → remember** turn loop;
- **three turns in one conversation** (each: raw → condensed → retrieved → answer), **plus** a side-by-side of turn 2 **without** vs **with** condensation — the contrast is the lesson.

This pattern — variously called **query condensation**, **condense-question**, **history-aware retrieval**, or **standalone-question rewriting** — was popularized in production by conversational retrieval chains (LangChain's `ConversationalRetrievalChain` and its condense-question step, since refactored into the `create_history_aware_retriever` helper). It rests on an academic line known as **conversational query rewriting**; a light reference is Elgohary et al., *CANARD* (EMNLP 2019), which framed turning a context-dependent question into a standalone one. The catchy names are practitioner vocabulary, not terms coined by any single paper — the rewrite-then-retrieve mechanism predates them.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic lexical retriever and the rule-based condenser are the offline default, so every cell executes and the output is reproducible. Set `RAG_REAL_EMBED=1` (with `sentence-transformers` installed) to opt into the real dense retriever — it changes only the printed scores; every raw query, condensed query, retrieved chunk, and answer line is identical. Set `OPENAI_API_KEY` to see the real LLM condenser banner; the code always falls through to the deterministic rewriter so it stays reproducible.

## Setup

Two imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the lexical retriever's tokenizer and the condenser's rule matching. The default retriever is pure standard library — `numpy` is imported only inside the optional real path, so this file runs with **no** third-party package unless you opt into `RAG_REAL_EMBED=1`.

In [ ]:
import os
import re

print("ready — no network, no API key, no third-party dependency required")

## Step 0 — The support knowledge base (carried over from Parts 6–12)

The same support corpus as the earlier parts — refunds, the **E-4042** error code, shipping, warranty — so this part feels continuous. Short docs, so each doc is its own chunk (Part 5).

Two clauses are **new**, and they are what make the contrast real:

- **Chunk index 1** is the damaged-items **refund** — turn 2's real target. It says *"refund"*, so **only** the condensed *"refund policy for damaged items"* retrieves it; the raw *"what about damaged items?"* (no "refund") misses.
- **Chunk index 3** is a deliberate **distractor**: damaged goods by misuse, **no** refund. It is shorter and shares *"damaged"* with the raw fragment, so the un-condensed query lands **here** — the wrong, no-refund answer. That is the contrast's miss.

In [ ]:
KNOWLEDGE_BASE = [
    "Our refund window is 30 days from purchase, as long as the product is unused and in its original packaging.",
    "Merchandise that arrives damaged qualifies for a full refund even outside the usual window; email a photo to support@example.com.",
    "We process a refund back to your original card within five business days of receiving the return.",
    "Damaged goods caused by customer misuse are not covered and must be replaced at full price.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} support chunks.")

## Step 1 — Retrieval, with a transparent deterministic default

The default is a pure **lexical** retriever: score each chunk by content-word **overlap** with the query, with a tiny cosine-style normalization. It is crude, but deterministic, model-free, network-free, and enough to make the right chunk win — so the demo's output is reproducible. The real `sentence-transformers` path (Part 6's embedder) is one env flag away and changes **only** the printed scores.

It is also **why condensation matters here**: a lexical retriever can only match words the query actually contains, so a follow-up that drops the word *"refund"* cannot find a refund chunk until the condenser puts *"refund"* back in.

First the tokenizer: lowercase content tokens, with grammatical stop-words dropped so a query's *content* words (refund, window, damaged, items) drive the score instead of *"what / about"*, and a crude plural-`s` stemmer so `items`/`item` hash to the same content word. A real model needs neither hint.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")

# Grammatical words the lexical retriever ignores so a query's CONTENT words
# (refund, window, damaged, items) drive the score instead of "what / about".
_STOPWORDS = {
    "what", "whats", "is", "are", "the", "of", "a", "an", "for", "on", "in",
    "to", "how", "do", "does", "and", "my", "i", "there", "with", "your", "our",
    "about", "that", "it", "this", "long", "take", "takes", "s",
}


def _stem(tok):
    """Crudest possible stemmer: drop a trailing plural 's' so 'items' and
    'item', or 'refunds' and 'refund', hash to the same content word. A real
    model needs no such hint; the lexical stand-in does."""
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    """Lowercase content tokens, stop-words dropped, plural 's' stemmed."""
    return [_stem(t) for t in _TOKEN_RE.findall(text.lower()) if t not in _STOPWORDS]


print(_tokens("What about damaged items?"))

### The lexical retriever (offline default) and the real dense path

The retriever scores `overlap / sqrt(|q| * |c|)` — a cosine-flavored overlap kept in a readable `[0, 1]` band. Crude, but it puts *"refund window"* next to the refund chunk and *"refund policy for damaged items"* next to the damaged-items clause without a 90 MB download or a network call.

`load_real_retriever()` tries the real `sentence-transformers` model first and falls back transparently to the lexical stand-in, printing exactly once. The lexical retriever is the **default** so this notebook's output is reproducible whether or not a model happens to be cached — the same reason the condenser (Step 4) defaults to the rule rewriter. Set `RAG_REAL_EMBED=1` to opt into the dense path; it changes only the printed scores.

In [ ]:
class _LexicalRetriever:
    """Deterministic, model-free stand-in for a dense retriever (Part 2/Part 6).

    Score = (overlap of query content words with the chunk) normalized by the
    geometric mean of the two token counts -- a cosine-flavored overlap that
    keeps scores in a readable [0, 1] band. Crude, but it puts 'refund window'
    next to the refund chunk and 'refund policy for damaged items' next to the
    damaged-items clause without a 90 MB download or a network call."""

    def __init__(self, corpus):
        self.chunks = list(corpus)
        self._chunk_tokens = [set(_tokens(c)) for c in self.chunks]

    def _score(self, q_tokens, c_tokens):
        if not q_tokens or not c_tokens:
            return 0.0
        overlap = len(q_tokens & c_tokens)
        # cosine-style normalization: overlap / sqrt(|q| * |c|), in [0, 1].
        denom = (len(q_tokens) * len(c_tokens)) ** 0.5
        return overlap / denom

    def retrieve(self, query, k=1):
        q_tokens = set(_tokens(query))
        scored = [
            (self.chunks[i], self._score(q_tokens, self._chunk_tokens[i]))
            for i in range(len(self.chunks))
        ]
        scored.sort(key=lambda x: -x[1])
        return scored[:k]


def load_real_retriever(corpus):
    """Use a real sentence-transformers retriever; fall back transparently.

    The deterministic lexical retriever is the DEFAULT so this file's output is
    reproducible whether or not a model happens to be cached -- the same reason
    the condenser defaults to the rule rewriter. Set RAG_REAL_EMBED=1 to opt into
    the real dense path (it changes only the printed scores; the condensed
    queries and answers are identical). The fallback prints exactly once so the
    demo banner stays clean.
    """
    if not os.environ.get("RAG_REAL_EMBED"):
        if not load_real_retriever._announced:
            print("[embed] using deterministic lexical retriever (offline default; "
                  "set RAG_REAL_EMBED=1 for sentence-transformers)")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)
    try:
        from sentence_transformers import SentenceTransformer  # REAL path
        import numpy as _np

        model = SentenceTransformer("all-MiniLM-L6-v2")          # 384 dims

        class _DenseRetriever:
            def __init__(self):
                self.chunks = list(corpus)
                self.vectors = _np.asarray(
                    model.encode(self.chunks, normalize_embeddings=True))

            def retrieve(self, query, k=1):
                q = _np.asarray(model.encode([query], normalize_embeddings=True))[0]
                scores = self.vectors @ q
                top = _np.argsort(-scores)[:k]
                return [(self.chunks[i], float(scores[i])) for i in top]

        if not load_real_retriever._announced:
            print("[embed] using sentence-transformers (all-MiniLM-L6-v2)")
            load_real_retriever._announced = True
        return _DenseRetriever()
    except Exception as exc:  # not installed / no weights / offline
        if not load_real_retriever._announced:
            print(f"[embed] sentence-transformers unavailable ({type(exc).__name__}); "
                  "using deterministic lexical fallback")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)


load_real_retriever._announced = False
print("retriever ready.")

## Step 2 — Conversation memory: a rolling transcript of turns

This is the whole "memory": an ordered list of `(user, assistant)` turns the condenser reads to recover the topic a follow-up leaves implicit. Real systems **cap or summarize** it — the history grows without bound (a caveat we return to) — but here it stays small and we keep it whole so every line is eyeball-able. The loop appends to it **after** each answered turn, so the next follow-up can see the exchange.

In [ ]:
class Conversation:
    """An ordered list of (user, assistant) turns. The condenser reads it; the
    loop appends to it after each answered turn."""

    def __init__(self):
        self.turns = []                       # list of (user_text, assistant_text)

    def add(self, user_text, assistant_text):
        self.turns.append((user_text, assistant_text))

    def last_user(self):
        return self.turns[-1][0] if self.turns else ""

    def last_assistant(self):
        return self.turns[-1][1] if self.turns else ""

    def is_empty(self):
        return not self.turns


print("Conversation memory ready.")

## Step 3 — `generate()`: the real LLM-driven condenser (reference shape)

In production the condenser **is** an LLM: you hand it the conversation history and the new follow-up, and it returns a single standalone query. We show that prompt shape here and keep `generate()` as one swappable provider call — **OpenAI active**, Ollama and a `claude-opus-4-8` variant in comments (the repo convention). The offline demo never calls it: `build_condense_prompt()` documents the shape, and the deterministic condenser in Step 4 is the source of truth.

> SDK names and model ids move fast; check current docs. Only `generate()` needs edits to light up the real path.

In [ ]:
CONDENSE_SYSTEM = """You rewrite a user's latest message into a STANDALONE search query.
Given the conversation history and a follow-up that may rely on it (pronouns
like "that"/"it", or ellipsis like "what about ...?"), output ONE self-contained
query that needs no history to retrieve on. Resolve every pronoun and fill in the
implicit topic from the history. If the follow-up is ALREADY standalone, return
it unchanged. Output only the rewritten query, nothing else."""


def build_condense_prompt(history_text, follow_up):
    """The prompt a REAL LLM condenser would see: history + the new follow-up."""
    history = history_text if history_text else "(no prior turns)"
    return (f"{CONDENSE_SYSTEM}\n\nConversation so far:\n{history}\n\n"
            f"Follow-up message: {follow_up}\n\nStandalone query:")


def generate(prompt):
    """REAL path: ask a hosted LLM to condense the follow-up. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small, cheap chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,                              # deterministic, not creative
    )
    return resp.choices[0].message.content


# Local, zero-cost alternative (Ollama). Swap this in for generate() above:
#
# def generate(prompt):
#     import requests
#     r = requests.post(
#         "http://localhost:11434/api/generate",
#         json={"model": "llama3.1", "prompt": prompt, "stream": False},
#     )
#     return r.json()["response"]


# Anthropic / Claude alternative. Swap this in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY from the env
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,                            # required by the Messages API
#         messages=[{"role": "user", "content": prompt}],
#     )                                               # (no temperature: removed on Opus 4.8)
#     return resp.content[0].text


print(build_condense_prompt("", "What about damaged items?")[:80], "...")

## Step 4 — The deterministic condenser (the source of truth)

Given the conversation history and the latest raw follow-up, `condense()` returns `(standalone_query, note)` — where `note` explains the rewrite so the trace can print *why* the query came out the way it did. This is the offline default: a transparent rule rewriter you can read, exactly mirroring Part 15's `classify_complexity` and Part 19's controller (deterministic-here / trained-in-production). A real system swaps this body for one `generate()` call against `build_condense_prompt()`; the surrounding loop is identical.

The rules key on the **same cheap signals an LLM would weigh**:

- the follow-up is **already standalone** (a full question carrying its own topic) → **leave it alone** (do **not** pollute a fresh question with a stale topic — the "don't condense a fresh question" caveat, enforced);
- an elliptical **"what about ___?"** → splice the history **topic** in front (*"what about damaged items?"* → *"refund policy for damaged items"*);
- a pronoun **"that"/"it"** → resolve it to the history topic (*"how long does that refund take?"* → *"how long does the refund take to process"*).

`_topic_from_history` recovers the salient noun (here: *"refund"*) from the most recent turns — the one piece a follow-up leaves implicit. Note the ellipsis/pronoun signals are tested **before** the standalone check, because a question can mention the topic word and **still** dangle: *"how long does that refund take?"* says *"refund"* yet *"that"* still needs resolving. The under-rewrite guard passes the raw fragment through when no topic is recoverable, rather than inventing one.

In [ ]:
# Topics the rewriter knows how to splice/resolve, longest first so "refund
# window" would win over "refund" if both appeared. Drawn from the KB's nouns.
_KNOWN_TOPICS = ("refund", "warranty", "shipping", "payment", "order")

# A follow-up that already contains one of these is treated as standalone: it
# carries its own topic, so splicing history would only pollute it (a caveat).
_STANDALONE_MARKERS = _KNOWN_TOPICS


def _topic_from_history(conversation):
    """Recover the salient topic the follow-up leaves implicit.

    Scan recent turns (latest first) for a known topic noun. A real LLM infers
    this for free; the rule rewriter looks for the one the corpus uses. Returns
    the topic string, or "" if none is found (then we cannot safely condense)."""
    for user_text, assistant_text in reversed(conversation.turns):
        blob = f"{user_text} {assistant_text}".lower()
        for topic in _KNOWN_TOPICS:
            if topic in blob:
                return topic
    return ""


# Match an elliptical follow-up "what about <X>?" and capture <X> ("damaged
# items"). This is the ellipsis case: the user dropped the verb/topic entirely.
_WHAT_ABOUT_RE = re.compile(r"^\s*(?:and\s+)?what about\s+(.+?)\s*\??\s*$", re.IGNORECASE)

# Pronouns that, in a follow-up, point back at the history topic (coreference).
_PRONOUN_RE = re.compile(r"\b(that|it|this|those|these)\b", re.IGNORECASE)


def condense(conversation, follow_up):
    """Rewrite a follow-up into a standalone query using the history.

    Returns (standalone_query, note). `note` is a short, human-readable reason
    that the trace prints so you can SEE why the query came out the way it did.
    """
    raw = follow_up.strip()
    low = raw.lower()

    # --- Turn 1 / no history: nothing to condense against. -------------------
    if conversation.is_empty():
        return raw, "already standalone -- no rewrite"

    # We condense a follow-up when it is CONTEXT-DEPENDENT: an ellipsis
    # ("what about ...?") or a dangling pronoun ("that"/"it"). We test those
    # signals BEFORE the standalone check, because a question can mention the
    # topic word and STILL dangle -- "how long does that refund take?" says
    # "refund" yet "that" still needs resolving.
    is_ellipsis = bool(_WHAT_ABOUT_RE.match(raw))
    has_pronoun = bool(_PRONOUN_RE.search(raw))

    # --- Already standalone: carries its own topic AND has no dangling
    #     pronoun/ellipsis. Leave it alone -- splicing a stale topic into a
    #     fresh question would pollute the retrieval (the "don't condense a
    #     fresh question" caveat, enforced). ---------------------------------
    if not is_ellipsis and not has_pronoun and any(t in low for t in _STANDALONE_MARKERS):
        return raw, "already standalone -- no rewrite"

    topic = _topic_from_history(conversation)
    if not topic:
        # Under-rewrite guard: with no recoverable topic we cannot safely fill
        # the blank, so we pass the raw fragment through rather than invent one.
        return raw, "no topic in history -- left as-is (would need clarification)"

    # --- Ellipsis: "what about <X>?" -> "<topic> policy for <X>". ------------
    if is_ellipsis:
        tail = _WHAT_ABOUT_RE.match(raw).group(1).strip().rstrip("?")
        standalone = f"{topic} policy for {tail}"
        return standalone, f"spliced topic '{topic}' from history"

    # --- Coreference: a pronoun ("that"/"it") -> the history topic. ----------
    if has_pronoun:
        pron = _PRONOUN_RE.search(raw).group(0)
        # Replace the FIRST pronoun with the topic noun, then tidy: drop a
        # leading "and", a trailing "?", and any doubled topic word the user
        # already supplied ("that refund" -> "<topic> refund" -> "the refund").
        resolved = _PRONOUN_RE.sub(topic, raw, count=1)
        resolved = re.sub(r"^\s*and\s+", "", resolved, flags=re.IGNORECASE)
        resolved = re.sub(rf"\b{topic}\s+{topic}\b", f"the {topic}", resolved,
                          flags=re.IGNORECASE)
        resolved = resolved.strip().rstrip("?")
        return resolved, f"resolved '{pron}' -> {topic}"

    # --- Fallback: prepend the topic so retrieval at least sees it. ----------
    return f"{topic} {raw.rstrip('?')}", f"prepended topic '{topic}'"


# A quick eyeball of all three rewrites against a one-turn history:
_demo = Conversation()
_demo.add("What's our refund window?", "Our refund window is 30 days from purchase.")
for q in ("What's our refund window?", "What about damaged items?",
          "And how long does that refund take to process?"):
    print(f"{q!r:55} -> {condense(_demo, q)}")

## Step 5 — Grounded answer, then one conversational turn

`answer_from()` is the grounded extractive stand-in (Part 6's grounding): quote the single best-retrieved chunk so the answer is visibly tied to a source. The real path is one swappable hosted-LLM call.

`converse_turn()` is the whole conversational pipeline in one function. Each turn: read memory, **condense** the raw follow-up into a standalone query, **retrieve on the standalone query** (never the raw fragment), **answer** from the chunk, then **append** the turn to memory so the next follow-up can see it.

In [ ]:
def answer_from(retrieved):
    """Grounded extractive generate(): quote the single best-retrieved chunk."""
    if not retrieved:
        return "I don't know based on the available sources."
    best_text, _score = retrieved[0]
    return best_text


def converse_turn(conversation, store, raw_query, trace=True):
    """Run one turn and return (standalone_query, chunk, score, answer)."""
    standalone, note = condense(conversation, raw_query)
    text, score = store.retrieve(standalone, k=1)[0]
    answer = answer_from([(text, score)])

    if trace:
        print(f"  user (raw):   {raw_query}")
        print(f"  condensed:    {standalone}   [{note}]")
        print(f"  retrieved (score={score:.2f}): {text}")
        print(f"  assistant:    {answer}")

    # REMEMBER: append AFTER answering so the next turn sees this exchange.
    conversation.add(raw_query, answer)
    return standalone, text, score, answer


print("converse_turn ready.")

## Step 6 — Build the store and pick the condenser

Build the retriever store over the knowledge base (this prints the embed banner once), then announce which condenser is active. With no `OPENAI_API_KEY` we use the deterministic rule rewriter — the offline default. With a key set, the real LLM condenser *would* drive `generate(build_condense_prompt(...))`, but the code still falls through to the deterministic rewriter so output stays reproducible.

In [ ]:
store = load_real_retriever(KNOWLEDGE_BASE)     # prints the embed banner

if os.environ.get("OPENAI_API_KEY"):
    print("[condenser] OPENAI_API_KEY set; the real LLM condenser would drive "
          "generate(build_condense_prompt(...)). Falling through to the "
          "deterministic rewriter so output stays reproducible.")
else:
    print("[condenser] no OPENAI_API_KEY; using deterministic rule-based "
          "condenser (offline default)")

print(f"\nKnowledge base: {len(KNOWLEDGE_BASE)} support chunks (refund window, "
      "damaged-items refunds, refund timing, the E-4042 error, shipping, warranty).")

## Step 7 — One conversation, three turns

Three turns sharing **one** memory. Watch the `condensed:` line:

- **Turn 1** *"What's our refund window?"* — already standalone, so the condenser leaves it alone; retrieves the 30-day window.
- **Turn 2** *"What about damaged items?"* — needs history. The ellipsis borrows *"refund"* from turn 1 → *"refund policy for damaged items"* → hits the damaged-items refund clause (chunk 1).
- **Turn 3** *"And how long does that refund take to process?"* — coreference. *"that"* resolves to *"refund"* → *"how long does the refund take to process"* → hits the refund-timing chunk.

The raw fragment never touches the index — the standalone query does.

In [ ]:
chat = Conversation()

print("TURN 1")
converse_turn(chat, store, "What's our refund window?")

print("\nTURN 2")
converse_turn(chat, store, "What about damaged items?")

print("\nTURN 3")
converse_turn(chat, store, "And how long does that refund take to process?")

## Step 8 — The contrast: turn 2 without vs with condensation

This is the lesson. We re-run turn 2's follow-up against a memory holding **only** turn 1, so the history topic is exactly *"refund"* — isolating the condensation effect.

- **Without condensation** we send the **raw** fragment *"What about damaged items?"* to the index. It never says *"refund"*, so the lexical retriever can only match *"items"/"damaged"* — and lands on the **distractor** (chunk 3: damaged goods by misuse, **no** refund). A miss.
- **With condensation** we condense first, then retrieve on *"refund policy for damaged items"*. The spliced *"refund"* topic lands the query on the damaged-items refund clause (chunk 1). A hit.

Same index, same follow-up — the only difference is reading the conversation first.

In [ ]:
contrast_chat = Conversation()
contrast_chat.add(
    "What's our refund window?",
    "Refunds are accepted within 30 days of purchase, provided the item is "
    "unused and in its original packaging.")
follow_up = "What about damaged items?"
topic = _topic_from_history(contrast_chat)
print(f'Follow-up: "{follow_up}"  (history topic: {topic})')

# WITHOUT: send the RAW fragment to the index. It never says "refund", so a
# lexical retriever can only match "items"/"damaged" against the wrong chunk.
raw_text, raw_score = store.retrieve(follow_up, k=1)[0]
print("\n  WITHOUT condensation (retrieve the RAW follow-up):")
print(f"    retrieved (score={raw_score:.2f}): {raw_text}")
print("    -> MISS: 'damaged items' alone never mentions refunds, so the "
      "index returns the wrong chunk.")

# WITH: condense first, then retrieve on the standalone query.
standalone, _note = condense(contrast_chat, follow_up)
con_text, con_score = store.retrieve(standalone, k=1)[0]
print(f'\n  WITH condensation (retrieve "{standalone}"):')
print(f"    retrieved (score={con_score:.2f}): {con_text}")
print("    -> HIT: the spliced 'refund' topic lands the query on the "
      "damaged-items clause.")

## What you learned

- A one-shot retriever can't hold a conversation: follow-ups like *"What about damaged items?"* or *"how long does that take?"* are **context-dependent fragments** that retrieve nothing useful on their own.
- The fix is **query condensation** (history-aware query rewriting): before retrieval, rewrite the follow-up into a **self-contained, standalone query** using the conversation so far. The standalone query is what hits the index; the raw fragment never does.
- **Conversation memory** is just a rolling transcript of `(user, assistant)` turns the condenser reads to recover the implicit topic.
- The condenser handles **coreference and ellipsis**: a pronoun (*"that"* → the refund) and an elliptical *"what about ___?"* (splice *"refund"* in front) both resolve against the last topic in history. A genuinely fresh, standalone question is **left alone** — splicing a stale topic would pollute it.
- The condenser is **deterministic offline** (transparent rules you can read) and **LLM-driven in production** (`generate()` against `build_condense_prompt()`) — the same deterministic-here / trained-in-production split as Parts 15 and 19.
- The **contrast is the lesson**: same index, same follow-up — without condensation the raw fragment lands on the wrong chunk; with it, the condensed query hits the damaged-items refund clause.
- Everything ran **fully offline** and reproducibly; the dense-retriever and LLM-condenser paths are shown as the headline behind a flag, changing only the printed scores.

### Honest caveats (no invented numbers)

- **History window growth.** The transcript grows without bound; production **truncates or summarizes** it. Too much folded-in context dilutes retrieval with stale, off-topic words; too little drops the antecedent the current turn depends on.
- **Pronoun / coreference ambiguity.** Follow-ups are underspecified through pronouns and ellipsis; a **wrong antecedent** silently sends retrieval after the wrong entity (*which "it"?*).
- **Over- vs under-rewriting.** Over-rewriting drags in a topic the user abandoned and pulls retrieval off course; under-rewriting leaves a dangling pronoun so retrieval matches generic terms.
- **Topic switching.** On an abrupt subject change, conditioning on history is **harmful** — the rewriter must recognize a fresh, self-contained question and **not** condense it.
- **A new failure point on the critical path.** The rewrite is an extra model call that adds latency and can mangle the query; errors compound across rewrite → retrieve → read.

### Where the series goes next

You now have a chat-shaped pipeline: memory in, a standalone query out, the right chunk retrieved. Point the loop at your own corpus and conversations, watch where the condenser over- or under-rewrites or mis-resolves a pronoun, and tune the rules (or hand the rewrite to a real LLM) from there.